# 01 – Recogida de Datos (Data Collection)

**Responsable**: David (Data Analyst)  
**Fecha**: Mayo 2026  
**Objetivo**: Obtener ofertas de empleo del sector datos en España mediante scraping de portales de empleo.

## Fuentes evaluadas

| Portal | Método | Resultado |
|--------|--------|-----------|
| InfoJobs | requests + BeautifulSoup | Bloqueo (CAPTCHA) |
| InfoJobs | Playwright (navegador real) | Bloqueo (Distil Networks) |
| Indeed | requests + BeautifulSoup | Bloqueo (403 Security Check) |
| Indeed | Playwright visible | **Exitoso** |

**Conclusión**: Solo Indeed permite scraping si se utiliza un navegador real no headless.

## Consideraciones éticas y legales
- Se respeta `robots.txt` de Indeed en la medida de lo posible.
- Se utiliza un User-Agent realista y un retardo entre peticiones para no sobrecargar el servidor.
- Los datos extraídos se usan exclusivamente con fines educativos dentro de este proyecto.

### Investigación sobre la paginación de Indeed

Se analizó la posibilidad de recorrer varias páginas de resultados para aumentar el volumen de ofertas por búsqueda.

**Parámetro de paginación**  
- Indeed utiliza el parámetro `start` en la URL (ej: `&start=10` para la segunda página, `&start=20` para la tercera).  
- Cada página muestra aproximadamente 10-15 resultados visibles.

**Barrera de inicio de sesión**  
- Al intentar acceder a `start=10` o superior, el portal redirige a una pantalla de creación de cuenta o inicio de sesión.  
- No se trata de un CAPTCHA ni un bloqueo HTTP; es un muro de autenticación obligatorio.

**Valoración ética y de riesgos**  
- La automatización con cuenta real está expresamente prohibida en los Términos de Uso de Indeed y podría conllevar la suspensión de la cuenta.  
- Aunque técnicamente sería posible iniciar sesión mediante Playwright, el equipo ha decidido **no implementar esta vía** por coherencia con las buenas prácticas profesionales y el respeto a las condiciones del portal.  
- Esta limitación se documenta como un sesgo de recolección: los datos proceden exclusivamente de la primera página de resultados de cada búsqueda, lo que podría favorecer ofertas más recientes o promocionadas.

**Estrategia adoptada**  
- En lugar de profundizar verticalmente, se amplía la cobertura horizontal: se ejecutan múltiples búsquedas con distintos roles y ubicaciones para maximizar el número de ofertas únicas desde la primera página de cada consulta.  
- Se complementa con la extracción de la descripción completa desde la página de detalle de cada oferta, lo que enriquece significativamente el dataset sin necesidad de paginación.

## Cómo ejecutar el scraping

La función `scrape_indeed()` ya está definida más abajo y lista para usarse.

**Ejemplo 1 – Búsqueda simple**  
Ejecuta la siguiente celda para obtener ofertas de "analista de datos" en Madrid:

```python
df = await scrape_indeed("analista de datos", "Madrid", headless=False)
df.head()
```

**Ejemplo 2 – Múltiples búsquedas en lote**  
Para buscar varios perfiles a la vez, añade una celda como esta:

```python
busquedas = [
    ("data scientist", "Madrid"),
    ("data engineer", "Barcelona"),
    ("machine learning", "Valencia"),
]
for kw, loc in busquedas:
    await scrape_indeed(keyword, location, headless=False)
```

Todas las ofertas se acumularán automáticamente en  
`data/raw/scraped_jobs.csv`, sin duplicados.
```

Ejecútala para que se renderice.

In [9]:
import asyncio
import pandas as pd
import os
from playwright.async_api import async_playwright

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DEFAULT_OUTPUT = "scraped_jobs.csv"


async def scrape_indeed(
    keyword,
    location,
    output_file=DEFAULT_OUTPUT,
    headless=False
):

    base_url = "https://es.indeed.com/jobs"
    query = f"q={keyword.replace(' ','+')}&l={location.replace(' ','+')}"
    url = f"{base_url}?{query}"

    print(f"Buscando: {keyword}")
    print(f"Ubicación: {location}")
    print(f"URL: {url}")

    async with async_playwright() as p:

        navegador = await p.chromium.launch(
            headless=headless
        )

        pagina = await navegador.new_page()

        await pagina.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60000
        )

        input(
            "\nResuelve cualquier verificación si aparece y pulsa Enter..."
        )

        # Esperar a que aparezcan resultados
        try:

            await pagina.wait_for_selector(
                "a.jcs-JobTitle",
                timeout=15000
            )

            print("Resultados detectados")

        except Exception:

            print("\nNo se detectaron ofertas")

            print("\nURL actual:")
            print(pagina.url)

            print("\nTítulo página:")
            print(await pagina.title())

            html = await pagina.content()

            print("\nPrimeros 1000 caracteres HTML:\n")
            print(html[:1000])

            await navegador.close()

            return pd.DataFrame()

        tarjetas = pagina.locator(
            ".job_seen_beacon"
        )

        tarjetas_count = await tarjetas.count()

        print(f"\nTarjetas encontradas: {tarjetas_count}")

        if tarjetas_count == 0:

            print("No hay resultados")

            await navegador.close()

            return pd.DataFrame()

        nuevas_ofertas = []

        for i in range(tarjetas_count):

            try:

                tarjeta = tarjetas.nth(i)

                titulo = None
                empresa = None
                ubicacion = None
                enlace = None
                descripcion = None
                metadatos = None

                # -------- TÍTULO --------

                titulo_elem = tarjeta.locator(
                    "a.jcs-JobTitle"
                )

                if await titulo_elem.count() > 0:

                    titulo = await titulo_elem.inner_text()

                    href = await titulo_elem.get_attribute(
                        "href"
                    )

                    if href:

                        enlace = (
                            "https://es.indeed.com" + href
                            if href.startswith("/")
                            else href
                        )

                # -------- EMPRESA --------

                empresa_elem = tarjeta.locator(
                    '[data-testid="company-name"]'
                )

                if await empresa_elem.count() > 0:

                    empresa = await empresa_elem.inner_text()

                # -------- UBICACIÓN --------

                ubicacion_elem = tarjeta.locator(
                    '[data-testid="text-location"]'
                )

                if await ubicacion_elem.count() > 0:

                    ubicacion = await ubicacion_elem.inner_text()

                # -------- METADATOS --------

                meta = tarjeta.locator(
                    '[data-testid="attribute_snippet_testid"]'
                )

                total_meta = await meta.count()

                lista_meta = []

                for j in range(total_meta):

                    texto = await meta.nth(j).inner_text()

                    lista_meta.append(texto)

                if lista_meta:

                    metadatos = " | ".join(lista_meta)

                # -------- DESCRIPCIÓN --------

                if await titulo_elem.count() > 0:

                    await titulo_elem.scroll_into_view_if_needed()

                    await titulo_elem.click()

                    try:

                        await pagina.locator(
                            "#jobDescriptionText"
                        ).wait_for(
                            timeout=10000
                        )

                        descripcion = await pagina.locator(
                            "#jobDescriptionText"
                        ).inner_text()

                    except:

                        descripcion = None

                print(
                    f"[{i+1}/{tarjetas_count}] {titulo}"
                )

                nuevas_ofertas.append({

                    "titulo": titulo,
                    "empresa": empresa,
                    "ubicacion": ubicacion,
                    "enlace": enlace,
                    "metadatos": metadatos,
                    "descripcion": descripcion

                })

            except Exception as e:

                print(
                    f"[{i+1}] Error: {e}"
                )

        await navegador.close()

    df_nuevas = pd.DataFrame(
        nuevas_ofertas
    )

    print(
        f"\nOfertas extraídas: {len(df_nuevas)}"
    )

    raw_dir = os.path.join(
        PROJECT_ROOT,
        "data",
        "raw"
    )

    os.makedirs(
        raw_dir,
        exist_ok=True
    )

    filepath = os.path.join(
        raw_dir,
        output_file
    )

    if os.path.exists(filepath):

        df_existente = pd.read_csv(
            filepath
        )

        df_total = pd.concat(
            [df_existente, df_nuevas],
            ignore_index=True
        )

        df_total = df_total.drop_duplicates(
            subset=["enlace"]
        )

    else:

        df_total = df_nuevas

    df_total.to_csv(
        filepath,
        index=False,
        encoding="utf-8"
    )

    print(
        f"\nGuardado en:\n{filepath}"
    )

    print(
        f"Total registros: {len(df_total)}"
    )

    return df_total

In [10]:
df_test = await scrape_indeed(
    keyword="analista de datos",
    location="Madrid",
    headless=False
)

df_test.head()

Buscando: analista de datos
Ubicación: Madrid
URL: https://es.indeed.com/jobs?q=analista+de+datos&l=Madrid

No se detectaron ofertas

URL actual:
https://es.indeed.com/jobs?q=analista+de+datos&l=Madrid

Título página:


TargetClosedError: Page.title: Target page, context or browser has been closed

In [5]:
# Probar la función con "analista de datos" en Madrid
df = await scrape_indeed("analista de datos", "Madrid", headless=False)

# Mostrar las primeras filas
df.head()

Buscando: analista de datos en Madrid
URL: https://es.indeed.com/jobs?q=analista+de+datos&l=Madrid
Tarjetas visibles detectadas.
Tarjetas parseadas: 32
Nuevas ofertas extraídas: 32
Ofertas ya existentes en scraped_jobs.csv: 154
Total de ofertas únicas guardadas en /mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo2/1_Proyecto/proyecto-eda-roles-datos/data/raw/scraped_jobs.csv: 170


,titulo,empresa,ubicacion,enlace,metadatos
0,DPO Analista de Datos,SEK EDUCATION GROUP,"Villanueva de la Cañada, Madrid provincia",https://es.indeed.com/rc/clk?jk=6e24f34f410689...,Jornada completa+1
1,Analista de datos,domestiko.com,"Madrid, Madrid provincia",https://es.indeed.com/rc/clk?jk=f9672f50d90f3b...,Jornada completa+1 | De lunes a viernes
2,27762 - Analista de Datos,INECO,"28036 Madrid, Madrid provincia",https://es.indeed.com/rc/clk?jk=687fab71451875...,NaN
3,Beca Analista de datos y digitalización,TK Elevator,"28935 Móstoles, Madrid provincia",https://es.indeed.com/rc/clk?jk=993cf9b976de44...,Jornada completa
4,Analista de Datos/Power BI,Indra,"Teletrabajo in Madrid, Madrid provincia",https://es.indeed.com/rc/clk?jk=20a5d2f27315f8...,NaN


In [3]:
# Búsqueda múltiple de diferentes perfiles de datos en España
busquedas = [
    ("data scientist", "Madrid"),
    ("data engineer", "Barcelona"),
    ("machine learning", "Valencia"),
    # Puedes añadir más tuplas (keyword, location) aquí
]

for keyword, location in busquedas:
    print(f"\n{'='*50}")
    print(f"Iniciando búsqueda: {keyword} en {location}")
    await scrape_indeed(keyword, location, headless=False)
    print(f"Búsqueda de {keyword} completada.\n")

print("Todas las búsquedas finalizadas. Los datos están en data/raw/scraped_jobs.csv")


Iniciando búsqueda: data scientist en Madrid
Buscando: data scientist en Madrid
URL: https://es.indeed.com/jobs?q=data+scientist&l=Madrid
Tarjetas visibles detectadas.
Tarjetas parseadas: 32
Nuevas ofertas extraídas: 32
Ofertas ya existentes en scraped_jobs.csv: 79
Total de ofertas únicas guardadas en /mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo2/1_Proyecto/proyecto-eda-roles-datos/data/raw/scraped_jobs.csv: 94
Búsqueda de data scientist completada.


Iniciando búsqueda: data engineer en Barcelona
Buscando: data engineer en Barcelona
URL: https://es.indeed.com/jobs?q=data+engineer&l=Barcelona
Tarjetas visibles detectadas.
Tarjetas parseadas: 32
Nuevas ofertas extraídas: 32
Ofertas ya existentes en scraped_jobs.csv: 94
Total de ofertas únicas guardadas en /mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo2/1_Proyecto/proyecto-eda-roles-datos/data/raw/scraped_jobs.csv: 109
Búsqueda de data engineer completada.


Iniciando búsqueda: machine learnin

In [1]:
import requests

APP_ID = "048db8ad"
APP_KEY = "2d8003e1aeb760637f6d24e52b7bebf3"

url = "https://api.adzuna.com/v1/api/jobs/es/search/1"
params = {
    "app_id": APP_ID,
    "app_key": APP_KEY,
    "what": "data scientist",
    "where": "Madrid",
    "content-type": "application/json"
}

resp = requests.get(url, params=params)
print("Status:", resp.status_code)

if resp.status_code == 200:
    data = resp.json()
    total = data.get("count", 0)
    print(f"Total ofertas encontradas: {total}")
    for job in data.get("results", [])[:3]:
        print("-", job.get("title"))
        print("  Empresa:", job.get("company", {}).get("display_name"))
        print("  Ubicación:", job.get("location", {}).get("display_name"))
        sal_min = job.get("salary_min")
        sal_max = job.get("salary_max")
        currency = job.get("salary_currency", "EUR")
        print(f"  Salario: {sal_min} - {sal_max} {currency}")
        desc = job.get("description", "")
        print("  Descripción (primeros 200 chars):", desc[:200])
        print("  URL:", job.get("redirect_url"))
        print("---")
else:
    print("Error:", resp.text)

Status: 200
Total ofertas encontradas: 30
- Data Scientist
  Empresa: Nextail
  Ubicación: Madrid
  Salario: None - None EUR
  Descripción (primeros 200 chars): About the Role We are seeking a Data Scientist with experience to join our Data Science team. This new team member will play a crucial role in developing data science solutions from conception to impl
  URL: https://www.adzuna.es/details/5713965485?utm_medium=api&utm_source=048db8ad
---
- Data Scientist
  Empresa: Abound
  Ubicación: Madrid
  Salario: None - None EUR
  Descripción (primeros 200 chars): About Abound We’re redefining consumer lending in the UK, and beyond. Using advanced AI and Open Banking data, we make fair, affordable personal finance available to more people. While traditional len
  URL: https://www.adzuna.es/details/5725426005?utm_medium=api&utm_source=048db8ad
---
- Data Scientist
  Empresa: Encore Capital Group
  Ubicación: Madrid
  Salario: None - None EUR
  Descripción (primeros 200 chars): The role is e

In [2]:
import os
import requests
import time
import pandas as pd
from dotenv import load_dotenv

# Cargar automáticamente el .env que está en la raíz del proyecto
load_dotenv()

APP_ID = os.getenv("ADZUNA_APP_ID")
APP_KEY = os.getenv("ADZUNA_APP_KEY")

if not APP_ID or not APP_KEY:
    raise ValueError("Falta ADZUNA_APP_ID o ADZUNA_APP_KEY en el archivo .env")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DEFAULT_OUTPUT = "scraped_jobs.csv"

def scrape_adzuna(keyword, location, output_file=DEFAULT_OUTPUT, max_pages=5):
    """
    Busca ofertas en España usando la API de Adzuna y las guarda en CSV unificado.
    """
    base_url = "https://api.adzuna.com/v1/api/jobs/es/search"
    all_offers = []

    for page in range(1, max_pages + 1):
        params = {
            "app_id": APP_ID,
            "app_key": APP_KEY,
            "what": keyword,
            "where": location,
            "content-type": "application/json",
            "results_per_page": 50
        }
        resp = requests.get(f"{base_url}/{page}", params=params)
        
        if resp.status_code != 200:
            print(f"  [!] Error en página {page}: {resp.status_code}")
            break

        data = resp.json()
        offers = data.get("results", [])
        if not offers:
            print(f"  [i] No hay más resultados en página {page}. Finalizando.")
            break

        for job in offers:
            all_offers.append({
                "titulo": job.get("title"),
                "empresa": job.get("company", {}).get("display_name"),
                "ubicacion": job.get("location", {}).get("display_name"),
                "enlace": job.get("redirect_url"),
                "metadatos": f"{job.get('contract_type', '')} | {job.get('category', {}).get('label', '')}",
                "descripcion": job.get("description"),
                "salario_min": job.get("salary_min"),
                "salario_max": job.get("salary_max"),
                "salario_moneda": job.get("salary_currency", "EUR")
            })

        print(f"  Página {page}: {len(offers)} ofertas extraídas. Total acumulado: {len(all_offers)}")
        time.sleep(1.1)  # respetar rate limit

    if not all_offers:
        print("No se obtuvieron ofertas.")
        return pd.DataFrame()

    df_nuevas = pd.DataFrame(all_offers)
    print(f"Total nuevas ofertas: {len(df_nuevas)}")

    # Unificar con archivo existente
    raw_dir = os.path.join(PROJECT_ROOT, "data", "raw")
    os.makedirs(raw_dir, exist_ok=True)
    filepath = os.path.join(raw_dir, output_file)

    if os.path.exists(filepath):
        df_existente = pd.read_csv(filepath)
        print(f"Ofertas ya existentes: {len(df_existente)}")
        df_total = pd.concat([df_existente, df_nuevas], ignore_index=True)
        df_total = df_total.drop_duplicates(subset=["enlace"], keep="first")
    else:
        df_total = df_nuevas

    df_total.to_csv(filepath, index=False, encoding="utf-8")
    print(f"Total ofertas únicas guardadas: {len(df_total)}")
    return df_total

In [3]:
df_test = scrape_adzuna("data scientist", "Madrid", max_pages=2)
df_test[["titulo", "empresa", "descripcion"]].head()

  Página 1: 30 ofertas extraídas. Total acumulado: 30
  [i] No hay más resultados en página 2. Finalizando.
Total nuevas ofertas: 30
Total ofertas únicas guardadas: 30


,titulo,empresa,descripcion
0,Data Scientist,Nextail,About the Role We are seeking a Data Scientist...
1,Data Scientist,Abound,About Abound We’re redefining consumer lending...
2,Data Scientist,Encore Capital Group,The role is embedded day‑to‑day with the Spani...
3,Data Scientist,Vodafone,Build and deliver data science & gen AI soluti...
4,Data Scientist,Mática Partners,¿Te apasiona el BigData? ¿Quieres formar parte...
